In [ ]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get('HF2')
if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    print("Warning: HF2 secret not found in Colab Secrets.")

In [ ]:
!pip install autoawq -q
from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer

model_path = "google/gemma-2-2b-it"
quant_path = "google/gemma-2-2b-awq"
quant_config = {"zero_point": True, "q_group_size": 128, "w_bit": 4, "version": "GEMM"}

# pip install autoawq
def quantize_awq(save_dir="./minstral-awq-4bit"):
    from awq import AutoAWQForCausalLM
    from transformers import AutoTokenizer

    tok = AutoTokenizer.from_pretrained(model_path)
    model = AutoAWQForCausalLM.from_pretrained(model_path)

    quant_config = {"zero_point": True, "q_group_size": 128, "w_bit": 4, "version": "GEMM"}
    model.quantize(tok, quant_config=quant_config)

    # model.save_quantized(save_dir)
    # tok.save_pretrained(save_dir)
    return model, tok


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 7.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


/usr/local/lib/python3.12/dist-packages/awq/__init__.py:21: DeprecationWarning: 
I have left this message as the final dev message to help you transition.

Important Notice:
- AutoAWQ is officially deprecated and will no longer be maintained.
- The last tested configuration used Torch 2.6.0 and Transformers 4.51.3.
- If future versions of Transformers break AutoAWQ compatibility, please report the issue to the Transformers project.

Alternative:
- AutoAWQ has been adopted by the vLLM Project: https://github.com/vllm-project/llm-compressor

For further inquiries, feel free to reach out:
- X: https://x.com/casper_hansen_
- LinkedIn: https://www.linkedin.com/in/casper-hansen-804005170/

  warnings.warn(_FINAL_DEV_MESSAGE, category=DeprecationWarning, stacklevel=1)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes

In [ ]:
# from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer
import torch

# Load the quantized model and tokenizer
model, tokenizer = quantize_awq()
model.to("cuda")

# Inference

def chat():
    print("Chat started. Type 'exit' to stop.")
    while True:
        user_input = input("User: ")
        if user_input.lower() == 'exit':
            break

        # Simple chat template format
        input_ids = tokenizer(user_input, return_tensors="pt").input_ids.cuda()

        outputs = model.generate(
            input_ids,
            max_new_tokens=200,
            do_sample=True,
            temperature=0.7,
            top_p=0.95
        )

        response = tokenizer.decode(outputs[0][input_ids.shape[-1]:], skip_special_tokens=True)
        print(f"LLM: {response}")

chat()

NameError: name 'quantize_awq' is not defined

In [ ]:
#!/usr/bin/env python3
"""
benchmark_llm.py

Benchmarks a Hugging Face causal LM on:
  1. Speed    -> tokens/sec, latency, throughput, TTFT
  2. Memory   -> on-disk model size, peak GPU/CPU memory during inference
  3. Smartness-> MMLU, GSM8K, HumanEval, TruthfulQA, HellaSwag (via lm-eval)

Each run is written as a timestamped JSON file to --output-dir so you can
run this against multiple model variants (fp16, AWQ, GPTQ, etc.) and diff
the results afterwards.

Install:
    pip install torch transformers accelerate lm-eval

For HumanEval (executes model-generated code to check correctness), you
must explicitly opt in:
    export HF_ALLOW_CODE_EVAL=1

Usage:
    python benchmark_llm.py --model google/gemma-2-2b --tag fp16
    python benchmark_llm.py --model ./gemma-2-2b-awq-4bit --tag awq4bit
    python benchmark_llm.py --model ./gemma-2-2b-awq-4bit --tag awq4bit --skip-smartness
"""

import argparse
import gc
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path

import torch


# --------------------------------------------------------------------------- #
# Helpers
# --------------------------------------------------------------------------- #

def log(msg: str) -> None:
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}", flush=True)


def get_device() -> str:
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def dir_size_bytes(path: str) -> int:
    """Sum file sizes under a local directory. Returns 0 for a Hub repo id
    (not a local path) — disk size for those isn't meaningful pre-download."""
    p = Path(path)
    if not p.exists() or not p.is_dir():
        return 0
    return sum(f.stat().st_size for f in p.rglob("*") if f.is_file())


def bytes_to_gb(b: int) -> float:
    return round(b / (1024 ** 3), 3)


# --------------------------------------------------------------------------- #
# 1. Speed benchmark
# --------------------------------------------------------------------------- #

def benchmark_speed(model, tokenizer, device, prompts, max_new_tokens=128, warmup=1):
    """
    Measures:
      - TTFT: time to first generated token (prefill + first decode step)
      - per-request latency: total wall time for a full generation
      - tokens/sec: decode-phase throughput per request
      - throughput: total generated tokens / total wall time across all prompts
    """
    results = []

    # Warmup run(s) - not counted, lets CUDA kernels/caches settle
    for _ in range(warmup):
        inputs = tokenizer(prompts[0], return_tensors="pt").to(device)
        with torch.no_grad():
            model.generate(**inputs, max_new_tokens=16, do_sample=False)
    if device == "cuda":
        torch.cuda.synchronize()

    total_generated_tokens = 0
    total_wall_time = 0.0

    for i, prompt in enumerate(prompts):
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        input_len = inputs["input_ids"].shape[1]

        if device == "cuda":
            torch.cuda.synchronize()
        t_start = time.perf_counter()

        # TTFT: generate exactly 1 token to capture prefill + first decode step
        with torch.no_grad():
            first_out = model.generate(
                **inputs, max_new_tokens=1, do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        if device == "cuda":
            torch.cuda.synchronize()
        t_first_token = time.perf_counter()
        ttft = t_first_token - t_start

        # Full generation for latency / throughput
        with torch.no_grad():
            out = model.generate(
                **inputs, max_new_tokens=max_new_tokens, do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        if device == "cuda":
            torch.cuda.synchronize()
        t_end = time.perf_counter()

        latency = t_end - t_start
        gen_tokens = out.shape[1] - input_len
        tok_per_sec = gen_tokens / latency if latency > 0 else 0.0

        total_generated_tokens += gen_tokens
        total_wall_time += latency

        results.append({
            "prompt_index": i,
            "input_tokens": input_len,
            "generated_tokens": gen_tokens,
            "ttft_sec": round(ttft, 4),
            "latency_sec": round(latency, 4),
            "tokens_per_sec": round(tok_per_sec, 2),
        })
        log(f"  prompt {i}: TTFT={ttft:.3f}s  latency={latency:.3f}s  "
            f"{tok_per_sec:.1f} tok/s")

    throughput = total_generated_tokens / total_wall_time if total_wall_time > 0 else 0.0

    return {
        "per_prompt": results,
        "avg_ttft_sec": round(sum(r["ttft_sec"] for r in results) / len(results), 4),
        "avg_latency_sec": round(sum(r["latency_sec"] for r in results) / len(results), 4),
        "avg_tokens_per_sec": round(
            sum(r["tokens_per_sec"] for r in results) / len(results), 2
        ),
        "overall_throughput_tok_per_sec": round(throughput, 2),
    }


# --------------------------------------------------------------------------- #
# 2. Memory benchmark
# --------------------------------------------------------------------------- #

def benchmark_memory(model, model_path, device):
    on_disk_bytes = dir_size_bytes(model_path)

    param_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
    buffer_bytes = sum(b.numel() * b.element_size() for b in model.buffers())
    n_params = sum(p.numel() for p in model.parameters())

    mem = {
        "on_disk_size_gb": bytes_to_gb(on_disk_bytes) if on_disk_bytes else None,
        "in_memory_param_size_gb": bytes_to_gb(param_bytes + buffer_bytes),
        "num_parameters": n_params,
        "num_parameters_billions": round(n_params / 1e9, 3),
    }

    if device == "cuda":
        mem["gpu_allocated_gb"] = bytes_to_gb(torch.cuda.memory_allocated())
        mem["gpu_reserved_gb"] = bytes_to_gb(torch.cuda.memory_reserved())
        mem["gpu_peak_allocated_gb"] = bytes_to_gb(torch.cuda.max_memory_allocated())

    return mem


# --------------------------------------------------------------------------- #
# 3. Smartness benchmark (lm-eval-harness)
# --------------------------------------------------------------------------- #

DEFAULT_TASKS = {
    "mmlu": "mmlu",
    "gsm8k": "gsm8k",
    "humaneval": "humaneval",
    "truthfulqa": "truthfulqa_mc2",
    "hellaswag": "hellaswag",
}


def benchmark_smartness(model_path, tasks=None, limit=None, batch_size="auto"):
    """
    Runs lm-eval-harness tasks against the model.
    `limit` caps the number of examples per task (use e.g. 100 for a fast
    sanity pass; leave None for the full, publication-comparable score).
    """
    try:
        import lm_eval
        from lm_eval.models.huggingface import HFLM
    except ImportError:
        log("lm_eval not installed. Run: pip install lm-eval")
        return {"error": "lm_eval not installed"}

    tasks = tasks or list(DEFAULT_TASKS.values())

    if "humaneval" in tasks and os.environ.get("HF_ALLOW_CODE_EVAL") != "1":
        log("Skipping humaneval: set HF_ALLOW_CODE_EVAL=1 to allow code "
            "execution required for this benchmark.")
        tasks = [t for t in tasks if t != "humaneval"]

    lm = HFLM(pretrained=model_path, batch_size=batch_size)

    log(f"Running lm-eval tasks: {tasks} (limit={limit})")
    raw = lm_eval.simple_evaluate(model=lm, tasks=tasks, limit=limit)

    summary = {}
    for task_name, metrics in raw["results"].items():
        # keep only the primary numeric metrics, drop the "_stderr" clutter
        summary[task_name] = {
            k: v for k, v in metrics.items()
            if isinstance(v, (int, float)) and not k.endswith("_stderr,none")
        }
    return summary


# --------------------------------------------------------------------------- #
# Main
# --------------------------------------------------------------------------- #

DEFAULT_PROMPTS = [
    "Explain the difference between supervised and unsupervised learning.",
    "Write a short story about a robot learning to paint.",
    "What are the main causes of inflation in an economy?",
    "Describe how a binary search algorithm works.",
]


def main():
    parser = argparse.ArgumentParser(description="Benchmark an LLM: speed, memory, smartness")
    parser.add_argument("--model", required=True, help="HF model id or local path")
    parser.add_argument("--tag", default=None,
                         help="Label for this run, e.g. 'fp16' or 'awq4bit' (default: model name)")
    parser.add_argument("--output-dir", default="./benchmark_results")
    parser.add_argument("--max-new-tokens", type=int, default=128)
    parser.add_argument("--dtype", default="auto", choices=["auto", "float16", "bfloat16", "float32"])
    parser.add_argument("--skip-speed", action="store_true")
    parser.add_argument("--skip-memory", action="store_true")
    parser.add_argument("--skip-smartness", action="store_true")
    parser.add_argument("--smartness-tasks", nargs="*", default=None,
                         help="Subset of tasks, e.g. --smartness-tasks mmlu gsm8k")
    parser.add_argument("--smartness-limit", type=int, default=None,
                         help="Cap examples/task for a quick pass (omit for full eval)")
    args = parser.parse_args()

    tag = args.tag or Path(args.model).name
    device = get_device()
    log(f"Device: {device}")

    os.makedirs(args.output_dir, exist_ok=True)
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    out_path = Path(args.output_dir) / f"{tag}_{timestamp}.json"

    report = {
        "tag": tag,
        "model": args.model,
        "device": device,
        "timestamp_utc": timestamp,
    }

    # --- Load model once, reuse for speed + memory sections ---
    if not args.skip_speed or not args.skip_memory:
        from transformers import AutoModelForCausalLM, AutoTokenizer

        log(f"Loading model: {args.model}")
        dtype = None if args.dtype == "auto" else getattr(torch, args.dtype)
        tokenizer = AutoTokenizer.from_pretrained(args.model)
        model = AutoModelForCausalLM.from_pretrained(
            args.model,
            torch_dtype=dtype or "auto",
            device_map="auto" if device == "cuda" else None,
        )
        if device != "cuda":
            model = model.to(device)
        model.eval()

        if device == "cuda":
            torch.cuda.reset_peak_memory_stats()

        if not args.skip_speed:
            log("Running speed benchmark...")
            report["speed"] = benchmark_speed(
                model, tokenizer, device, DEFAULT_PROMPTS,
                max_new_tokens=args.max_new_tokens,
            )

        if not args.skip_memory:
            log("Running memory benchmark...")
            report["memory"] = benchmark_memory(model, args.model, device)

        del model
        gc.collect()
        if device == "cuda":
            torch.cuda.empty_cache()

    # --- Smartness: lm-eval loads/manages the model itself ---
    if not args.skip_smartness:
        log("Running smartness benchmarks (this can take a while)...")
        report["smartness"] = benchmark_smartness(
            args.model,
            tasks=args.smartness_tasks,
            limit=args.smartness_limit,
        )

    with open(out_path, "w") as f:
        json.dump(report, f, indent=2)

    log(f"Results written to {out_path}")
    print(json.dumps(report, indent=2))


if __name__ == "__main__":
    main()

In [ ]:
# ============================================================================
# Paste this whole cell into Colab and run it.
# It reads every *.json file in ./benchmark_results (or pass your own list)
# and renders the comparison charts directly in the cell output.
# ============================================================================

import glob
import json
import os

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="talk")
PALETTE = "viridis"

# ---- CONFIG: edit these two lines if needed -------------------------------
RESULTS_DIR = "./benchmark_results"   # folder containing your *.json runs
FILES = None                          # or set e.g. ["run1.json", "run2.json"]
# -----------------------------------------------------------------------------


def load_runs(results_dir, files):
    paths = files if files else sorted(glob.glob(os.path.join(results_dir, "*.json")))
    runs = []
    for p in paths:
        with open(p) as f:
            data = json.load(f)
        data["_source_file"] = os.path.basename(p)
        runs.append(data)
    if not runs:
        raise SystemExit(f"No JSON files found in {results_dir}. "
                          "Set RESULTS_DIR or FILES above.")
    return runs


def speed_dataframe(runs):
    rows = []
    for r in runs:
        s = r.get("speed")
        if not s:
            continue
        rows.append({
            "model": r["tag"],
            "Avg Tokens/sec": s["avg_tokens_per_sec"],
            "Avg Latency (s)": s["avg_latency_sec"],
            "Avg TTFT (s)": s["avg_ttft_sec"],
            "Throughput (tok/s)": s["overall_throughput_tok_per_sec"],
        })
    return pd.DataFrame(rows)


def memory_dataframe(runs):
    rows = []
    for r in runs:
        m = r.get("memory")
        if not m:
            continue
        row = {
            "model": r["tag"],
            "Param size (GB)": m.get("in_memory_param_size_gb"),
            "GPU allocated (GB)": m.get("gpu_allocated_gb"),
            "GPU peak (GB)": m.get("gpu_peak_allocated_gb"),
        }
        if m.get("on_disk_size_gb") is not None:
            row["On-disk size (GB)"] = m["on_disk_size_gb"]
        rows.append(row)
    return pd.DataFrame(rows)


def smartness_dataframe(runs):
    rows = []
    for r in runs:
        sm = r.get("smartness")
        if not sm:
            continue
        for task, metrics in sm.items():
            for metric_name, value in metrics.items():
                if not isinstance(value, (int, float)) or metric_name == "sample_len":
                    continue
                rows.append({
                    "model": r["tag"],
                    "task": f"{task} ({metric_name.split(',')[0]})",
                    "score": value,
                })
    return pd.DataFrame(rows)


def plot_speed(ax_group, df):
    metrics = ["Avg Tokens/sec", "Throughput (tok/s)", "Avg Latency (s)", "Avg TTFT (s)"]
    for ax, metric in zip(ax_group, metrics):
        if df.empty or metric not in df:
            ax.axis("off")
            continue
        sns.barplot(data=df, x="model", y=metric, hue="model", ax=ax,
                    palette=PALETTE, legend=False)
        ax.set_title(metric, pad=14)
        ax.set_xlabel("")
        ax.set_ylabel("")
        ax.set_ylim(top=df[metric].max() * 1.2)
        ax.bar_label(ax.containers[0], fmt="%.3f", padding=3)
        ax.tick_params(axis="x", rotation=20)


def plot_memory(ax, df):
    if df.empty:
        ax.axis("off")
        return
    value_cols = [c for c in df.columns if c != "model"]
    melted = df.melt(id_vars="model", value_vars=value_cols,
                      var_name="metric", value_name="GB").dropna(subset=["GB"])
    sns.barplot(data=melted, x="model", y="GB", hue="metric", ax=ax, palette=PALETTE)
    ax.set_title("Memory footprint")
    ax.set_xlabel("")
    ax.set_ylabel("GB")
    ax.tick_params(axis="x", rotation=20)
    ax.legend(title="", loc="upper right", fontsize=10)


def plot_smartness(ax, df):
    if df.empty:
        ax.axis("off")
        return
    sns.barplot(data=df, x="task", y="score", hue="model", ax=ax, palette=PALETTE)
    ax.set_title("Capability benchmarks (accuracy)")
    ax.set_xlabel("")
    ax.set_ylabel("Score")
    ax.set_ylim(0, 1)
    ax.tick_params(axis="x", rotation=20)
    ax.legend(title="", loc="upper right", fontsize=10)


# ---- Load + build dataframes -----------------------------------------------
runs = load_runs(RESULTS_DIR, FILES)
model_tags = [r["tag"] for r in runs]
print(f"Loaded {len(runs)} run(s): {model_tags}")

speed_df = speed_dataframe(runs)
memory_df = memory_dataframe(runs)
smartness_df = smartness_dataframe(runs)

# ---- Render dashboard inline (no files written) ----------------------------
fig = plt.figure(figsize=(18, 12))
gs = fig.add_gridspec(3, 4, height_ratios=[1, 1, 1.1])

speed_axes = [fig.add_subplot(gs[0, i]) for i in range(4)]
plot_speed(speed_axes, speed_df)

mem_ax = fig.add_subplot(gs[1, :2])
plot_memory(mem_ax, memory_df)

smart_ax = fig.add_subplot(gs[1:, 2:])
plot_smartness(smart_ax, smartness_df)

fig.suptitle(f"LLM Benchmark Comparison: {', '.join(model_tags)}",
             fontsize=18, fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()   # <- this is what makes it render in the Colab cell output

SystemExit: No JSON files found in ./benchmark_results. Set RESULTS_DIR or FILES above.

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
